# ReNewGenie: fine-tune on your own photos (Indian waste) and compare with TrashNet-only

Run in Google Colab with a T4 GPU (Runtime > Change runtime type).

**Not run by the author.** This notebook was written without a GPU. Run it yourself and quote only the numbers it prints.

## What you need
Upload a zip named `my_photos.zip` with this layout (see `docs/DATA_COLLECTION.md`):
```
my_photos/
  cardboard/ glass/ metal/ paper/ plastic/ trash/
    photo1.jpg ...
```
The notebook:
1. splits **your** photos 60/20/20 by photo (train/val/test),
2. fine-tunes MobileNetV2 (ImageNet weights) on your train photos,
3. reports accuracy and a confusion matrix on your held-out test photos only.

To compare with the demo's TrashNet-only models, test those models on the same held-out photos and compare the numbers honestly.

In [ ]:
import os, random, zipfile, numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from google.colab import files
SEED=7; random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
up=files.upload()   # choose my_photos.zip
zipfile.ZipFile(list(up)[0]).extractall('.')
CLASSES=['cardboard','glass','metal','paper','plastic','trash']
paths=[];labels=[]
for i,c in enumerate(CLASSES):
    d=os.path.join('my_photos',c)
    for f in sorted(os.listdir(d)):
        if f.lower().endswith(('.jpg','.jpeg','.png')): paths.append(os.path.join(d,f)); labels.append(i)
paths=np.array(paths); labels=np.array(labels)
print({c:int((labels==i).sum()) for i,c in enumerate(CLASSES)})

In [ ]:
idx=np.random.permutation(len(paths)); n=len(idx)
tr,va,te=idx[:int(.6*n)],idx[int(.6*n):int(.8*n)],idx[int(.8*n):]
def load(p):
    img=tf.io.decode_image(tf.io.read_file(p),channels=3,expand_animations=False)
    img=tf.image.resize(img,(224,224)); return tf.keras.applications.mobilenet_v2.preprocess_input(img)
def ds(ix,train):
    d=tf.data.Dataset.from_tensor_slices((paths[ix],labels[ix])).map(lambda p,l:(load(p),l),num_parallel_calls=tf.data.AUTOTUNE)
    if train: d=d.shuffle(512,seed=SEED)
    return d.batch(32).prefetch(tf.data.AUTOTUNE)
aug=keras.Sequential([layers.RandomFlip('horizontal'),layers.RandomRotation(.08),layers.RandomZoom(.15),layers.RandomBrightness(.2)])
base=keras.applications.MobileNetV2(include_top=False,weights='imagenet',input_shape=(224,224,3),pooling='avg'); base.trainable=False
inp=keras.Input((224,224,3)); x=aug(inp); x=base(x,training=False); x=layers.Dropout(.3)(x)
model=keras.Model(inp,layers.Dense(6,activation='softmax')(x))
model.compile(optimizer=keras.optimizers.Adam(1e-3),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
cb=[keras.callbacks.EarlyStopping(patience=4,restore_best_weights=True,monitor='val_accuracy')]
model.fit(ds(tr,True),validation_data=ds(va,False),epochs=15,callbacks=cb)
base.trainable=True
for l in base.layers[:-30]: l.trainable=False
model.compile(optimizer=keras.optimizers.Adam(1e-5),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
model.fit(ds(tr,True),validation_data=ds(va,False),epochs=10,callbacks=cb)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
pred=np.concatenate([model.predict(x,verbose=0).argmax(1) for x,_ in ds(te,False)])
print('Test photos:',len(te),'| accuracy:',(pred==labels[te]).mean().round(3))
print(confusion_matrix(labels[te],pred)); print(classification_report(labels[te],pred,target_names=CLASSES))
model.save('classifyWaste_finetuned.keras')